# Talk2Me Speech - Google Colab Setup

This notebook sets up the Talk2Me Speech environment in Google Colab and verifies all dependencies.

**Estimated time**: ~5 minutes for setup + GPU verification

## 1. Environment Setup

Clone the repository and install dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/your-org/talk2me-speech.git
%cd talk2me-speech

In [ ]:
# Install the package
!pip install -e . -q
print("✓ Installation complete")

## 2. Python and PyTorch Verification

Check Python version and PyTorch installation.

In [ ]:
import sys
import torch
import talk2me_speech

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Talk2Me Speech version: {talk2me_speech.__version__}")

## 3. GPU Availability

Check if GPU is available and show device details.

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU capability: {torch.cuda.get_device_capability(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    
    # Memory info
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU memory: {total_memory:.2f} GB")
    print(f"Allocated memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    print("⚠️  No GPU available. CPU-only mode.")
    print("For faster training, enable GPU in Colab: Runtime → Change runtime type → GPU")

## 4. Talk2Me Speech Package Verification

Verify that the package imports correctly and components are accessible.

In [ ]:
# Test core imports
try:
    from talk2me_speech.audio.normalize import normalize_audio
    from talk2me_speech.audio.resample import resample_audio
    from talk2me_speech.datasets.loader import load_manifest
    from talk2me_speech.datasets.manifest import write_manifest
    from talk2me_speech.evaluation.wer import compute_wer
    from talk2me_speech.models.registry import get_model_registry
    from talk2me_speech.models.whisper import WhisperModel
    from talk2me_speech.models.xlsr import XLSRModel
    
    print("✓ All core modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")

## 5. Audio Processing Test

Quick test of audio processing utilities.

In [ ]:
import numpy as np
from talk2me_speech.audio.normalize import normalize_audio

# Create test audio
test_audio = np.random.randn(16000).astype(np.float32)

# Normalize
normalized = normalize_audio(test_audio, target_dbfs=-20.0)

print(f"✓ Audio test passed")
print(f"  Input shape: {test_audio.shape}")
print(f"  Output shape: {normalized.shape}")
print(f"  Peak normalized: {np.max(np.abs(normalized)):.4f}")

## 6. Dataset and Manifest Test

Test manifest creation and loading.

In [ ]:
from talk2me_speech.datasets.manifest import write_manifest
from talk2me_speech.datasets.loader import load_manifest
import tempfile
import os

# Create sample records
sample_records = [
    {
        "id": "sample_001",
        "audio_path": "audio/sample1.wav",
        "transcript": "Chale yɛbɛ deploy no tomorrow",
        "duration": 2.5,
        "source": "test",
        "languages": ["en", "tw"]
    },
    {
        "id": "sample_002",
        "audio_path": "audio/sample2.wav",
        "transcript": "Accra is the capital",
        "duration": 1.8,
        "source": "test",
        "languages": ["en"]
    }
]

# Write to temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    manifest_path = os.path.join(tmpdir, "test.jsonl")
    write_manifest(manifest_path, sample_records)
    
    # Load it back
    loaded = load_manifest(manifest_path)
    
    print(f"✓ Manifest test passed")
    print(f"  Written: {len(sample_records)} records")
    print(f"  Loaded: {len(loaded)} records")
    print(f"\nSample record:")
    print(f"  ID: {loaded[0]['id']}")
    print(f"  Transcript: {loaded[0]['transcript']}")
    print(f"  Languages: {loaded[0]['languages']}")

## 7. Model Registry

List available models in the registry.

In [ ]:
from talk2me_speech.models.registry import get_model_registry

registry = get_model_registry()

print("Available models in registry:")
print()

whisper_models = {k: v for k, v in registry.items() if 'whisper' in k}
xlsr_models = {k: v for k, v in registry.items() if 'xlsr' in k}

print("Whisper models:")
for model_name, config in whisper_models.items():
    print(f"  - {model_name}: {config['architecture']}")

print()
print("XLS-R models:")
for model_name, config in xlsr_models.items():
    print(f"  - {model_name}: {config['architecture']}")

print(f"\n✓ Total models: {len(registry)}")

## 8. Evaluation Metrics

Test WER and CER calculation.

In [ ]:
from talk2me_speech.evaluation.wer import compute_wer
from talk2me_speech.evaluation.cer import compute_cer

# Test WER calculation
reference = "Chale yɛbɛ deploy no tomorrow"
hypothesis = "Chale we deploy tomorrow"

wer = compute_wer(reference, hypothesis)
cer = compute_cer(reference, hypothesis)

print("✓ Evaluation metrics test passed")
print(f"\nReference: {reference}")
print(f"Hypothesis: {hypothesis}")
print(f"WER: {wer:.4f}")
print(f"CER: {cer:.4f}")

# Test perfect match
perfect_wer = compute_wer(reference, reference)
print(f"\nPerfect match WER: {perfect_wer:.4f}")

## 9. Configuration System

Load and display configuration files.

In [ ]:
import yaml
import os

# Load base config
config_path = "configs/base.yaml"

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    print("✓ Configuration loaded successfully")
    print("\nBase configuration:")
    print(yaml.dump(config, default_flow_style=False))
else:
    print(f"❌ Config file not found: {config_path}")

## 10. Google Drive Integration (Optional)

Mount Google Drive for storing large datasets and checkpoints.

In [ ]:
# Uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Now you can use:
# # /content/drive/MyDrive/talk2me-speech/data/
# # /content/drive/MyDrive/talk2me-speech/models/

print("Google Drive integration is optional.")
print("")
print("To use Drive for datasets/checkpoints:")
print("  1. Uncomment the code above")
print("  2. Run this cell")
print("  3. Authorize access when prompted")
print("  4. Set data paths to /content/drive/MyDrive/...")

## 11. Run Unit Tests

Execute the test suite to verify everything works.

In [ ]:
import subprocess

# Run tests
result = subprocess.run(['python', '-m', 'pytest', 'tests/', '-v'], 
                       capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:")
    print(result.stderr)

## 12. System Information Summary

Final summary of the environment.

In [ ]:
import sys
import platform
import torch

print("=" * 60)
print("TALK2ME SPEECH - COLAB ENVIRONMENT SUMMARY")
print("=" * 60)

print(f"\nSystem:")
print(f"  Platform: {platform.system()} {platform.release()}")
print(f"  Python: {sys.version.split()[0]}")

print(f"\nMachine Learning Stack:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

print(f"\nTalk2Me Speech:")
import talk2me_speech
print(f"  Version: {talk2me_speech.__version__}")
print(f"  Status: ✓ Ready for research")

print(f"\nNext Steps:")
print(f"  1. Read README.md for project overview")
print(f"  2. Review notebooks/01_dataset_exploration.ipynb")
print(f"  3. Check configs/ for experiment setup")
print(f"  4. Run: python scripts/prepare_dataset.py")
print(f"  5. Follow experiment roadmap in README")

print("\n" + "=" * 60)